DEEP LEARNING Sistemas de recomendación basados en contenido

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

folder_path = '/content/drive/My Drive/Recomendadores'  # Ajusta si el nombre es diferente
os.listdir(folder_path)


['negocios.csv', 'test_reviews.csv', 'train_reviews.csv', 'usuarios.csv']

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Función para cargar los datos (asumiendo que tienes los archivos CSV)
def cargar_datos():
    try:
        # Cargar los diferentes conjuntos de datos

        usuarios_df = pd.read_csv(os.path.join(folder_path, 'usuarios.csv'))
        negocios_df = pd.read_csv(os.path.join(folder_path, 'negocios.csv'))
        train_reviews_df = pd.read_csv(os.path.join(folder_path, 'train_reviews.csv'))
        test_reviews_df = pd.read_csv(os.path.join(folder_path, 'test_reviews.csv'))

        #usuarios_df = pd.read_csv('usuarios.csv')
        #negocios_df = pd.read_csv('negocios.csv')
        #train_reviews_df = pd.read_csv('train_reviews.csv')
        #test_reviews_df = pd.read_csv('test_reviews.csv')


        print(f"Datos cargados correctamente:")
        print(f"- Usuarios: {usuarios_df.shape[0]} registros")
        print(f"- Negocios: {negocios_df.shape[0]} registros")
        print(f"- Train reviews: {train_reviews_df.shape[0]} registros")
        print(f"- Test reviews: {test_reviews_df.shape[0]} registros")

        return usuarios_df, negocios_df, train_reviews_df, test_reviews_df

    except FileNotFoundError as e:
        print(f"Error al cargar los archivos: {e}")
        return None, None, None, None

# Función para extraer características de usuarios
def extraer_caracteristicas_usuarios(usuarios_df):
    # Convertir la fecha a características numéricas
    usuarios_df['yelping_since'] = pd.to_datetime(usuarios_df['yelping_since'])
    usuarios_df['años_en_yelp'] = (datetime.now() - usuarios_df['yelping_since']).dt.days / 365

    # Características de interacción social
    usuarios_df['num_amigos'] = usuarios_df['friends'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() else 0)

    # Características de actividad
    usuarios_df['ratio_useful'] = usuarios_df['useful'] / (usuarios_df['review_count'] + 1)
    usuarios_df['ratio_funny'] = usuarios_df['funny'] / (usuarios_df['review_count'] + 1)
    usuarios_df['ratio_cool'] = usuarios_df['cool'] / (usuarios_df['review_count'] + 1)

    # Características de popularidad
    usuarios_df['popularidad'] = usuarios_df['fans'] / (usuarios_df['review_count'] + 1)

    # Suma total de cumplidos recibidos
    usuarios_df['total_compliments'] = (
        usuarios_df['compliment_hot'] + usuarios_df['compliment_more'] +
        usuarios_df['compliment_profile'] + usuarios_df['compliment_cute'] +
        usuarios_df['compliment_list'] + usuarios_df['compliment_note'] +
        usuarios_df['compliment_plain'] + usuarios_df['compliment_cool'] +
        usuarios_df['compliment_funny'] + usuarios_df['compliment_writer'] +
        usuarios_df['compliment_photos']
    )

    return usuarios_df

# Función para extraer características de negocios
def extraer_caracteristicas_negocios(negocios_df):

    # Extraer número de categorías
    negocios_df['num_categorias'] = negocios_df['categories'].apply(
        lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() else 0
    )

    return negocios_df


# Función para unificar los datasets
def unificar_datasets(usuarios_df, negocios_df, train_reviews_df, test_reviews_df):
    # Para el conjunto de entrenamiento
    print("Unificando conjunto de entrenamiento...")
    train_completo = train_reviews_df.copy()

    # Agregar características de usuario
    caracteristicas_usuario = [
        'review_count', 'useful', 'funny', 'cool', 'fans', 'average_stars',
        'años_en_yelp', 'num_amigos', 'total_compliments', 'ratio_useful',
        'ratio_funny', 'ratio_cool', 'popularidad'
    ]

    train_completo = pd.merge(
        train_completo,
        usuarios_df[['user_id'] + caracteristicas_usuario],
        on='user_id',
        how='left',
        suffixes=('', '_usuario')
    )

    # Agregar características del negocio
    caracteristicas_negocio = [
        'stars', 'review_count', 'is_open', 'num_categorias', 'address' , 'city',
        'state', 'postal_code', 'latitude', 'longitude' , 'attributes','is_open',
        'categories', 'hours'
    ]

    train_completo = pd.merge(
        train_completo,
        negocios_df[['business_id'] + caracteristicas_negocio],
        on='business_id',
        how='left',
        suffixes=('', '_negocio')
    )

    # Renombrar columnas para evitar confusión
    train_completo.rename(columns={
        'stars_negocio': 'promedio_estrellas_negocio',
        'stars': 'estrellas_review',
        'review_count_usuario': 'num_reviews_usuario',
        'review_count_negocio': 'num_reviews_negocio'
    }, inplace=True)

    # Para el conjunto de prueba
    print("Unificando conjunto de prueba...")
    test_completo = test_reviews_df.copy()

    # Conseguir user_id y business_id para cada review_id en test
    # Necesitamos esta información de train_reviews o alguna otra fuente
    id_mapping = train_reviews_df[['review_id', 'user_id', 'business_id']]
    test_completo = pd.merge(
        test_completo,
        id_mapping,
        on='review_id',
        how='left',
    )

    test_completo = test_completo.rename(columns={'user_id_x': 'user_id'})
    test_completo = test_completo.rename(columns={'business_id_x': 'business_id'})

    test_completo.drop(['business_id_y', 'user_id_y'], axis=1, inplace=True)


    print(test_completo.head())
    print(usuarios_df.head())
    # Agregar las mismas características que para el conjunto de entrenamiento
    test_completo = pd.merge(
        test_completo,
        usuarios_df[['user_id'] + caracteristicas_usuario],
        on='user_id',
        how='left',
        suffixes=('', '_usuario')
    )
    test_completo = pd.merge(
        test_completo,
        negocios_df[['business_id'] + caracteristicas_negocio],
        on='business_id',
        how='left',
        suffixes=('', '_negocio')
    )

    # Renombrar columnas igual que en entrenamiento
    test_completo.rename(columns={
        'stars_negocio': 'promedio_estrellas_negocio',
        'stars': 'estrellas_review',
        'review_count_usuario': 'num_reviews_usuario',
        'review_count_negocio': 'num_reviews_negocio'
    }, inplace=True)

    return train_completo, test_completo

In [ ]:
usuarios_df, negocios_df, train_reviews_df, test_reviews_df = cargar_datos()

<ipython-input-3-512311dc537a>:10: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  usuarios_df = pd.read_csv(os.path.join(folder_path, 'usuarios.csv'))


Datos cargados correctamente:
- Usuarios: 699619 registros
- Negocios: 30069 registros
- Train reviews: 967784 registros
- Test reviews: 414765 registros


In [ ]:
# Preprocesar datos
print("Extrayendo características de usuarios...")
usuarios_df = extraer_caracteristicas_usuarios(usuarios_df)

Extrayendo características de usuarios...


In [ ]:
print("Extrayendo características de negocios...")
negocios_df = extraer_caracteristicas_negocios(negocios_df)

Extrayendo características de negocios...


In [ ]:
# Unificar datasets
train_completo, test_completo = unificar_datasets(
    usuarios_df, negocios_df, train_reviews_df, test_reviews_df
)

Unificando conjunto de entrenamiento...
Unificando conjunto de prueba...
prueba3
                review_id                 user_id             business_id  \
0  ieYPmCImINjPzTDFmEKBKA  79F9QrQSet-b1yRCIM243Q  sXSUzImYOcRRI3xtG2M85g   
1  QIkJ8fZ4yx_QaHahWWszAA  chuM6TBkFHtTwJ6z96Hj1A  Ipt9ga67vVC_2ob3YmVwNA   
2  seR2KhblYMWg-k9zzN6aYA  hF68a0mpu97u0oaryFYhyg  _RG4IByyBR528CMc7DefJA   
3  BToo00Fi5pfJFA5MI2HM5g  G4yX5Q1tFfwSucFOmiyjdA  xxlbRiWWQkk-6LST3Hd12g   
4  FHJAzi1imodBit3RWK7zQA  Srqi1xb7exdB9uRHxDeEkw  LgGqdFLD7-ca0Z9F_q4Fuw   

   useful  funny  cool                                               text  \
0       1      0     1  Amazing coffee and chill atmosphere. The staff...   
1       4      0     2  I pass by this joint every time I make a run t...   
2       2      0     0  Came here when my kitten got very sick by the ...   
3       2      0     0  So I'll preface by saying we did have an overa...   
4       0      0     0  This place is a joke. Worst bar service ever. .

In [ ]:
# Reducir al 20%
train_reducido = train_completo.sample(frac=0.2, random_state=42)  #20% aleatorio reproducible

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd

# Si usas una GPU y PyTorch
model = SentenceTransformer('paraphrase-MiniLM-L3-v2')
model = model.to('cuda')  # Mover a GPU si está disponible


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def generar_embeddings(df, model, columna='text', batch_size=128):
    print("Generando embeddings para texto...")
    df[columna] = df[columna].fillna('')
    textos = df[columna].astype(str).tolist()

    embeddings = []
    for i in range(0, len(textos), batch_size):
        batch = textos[i:i+batch_size]
        batch_embeddings = model.encode(batch)
        embeddings.extend(batch_embeddings)

    df['text_embedding'] = embeddings
    return df


In [ ]:
def process_categories(categories):
    if pd.isna(categories) or not categories:
        return []

    if isinstance(categories, str):
        try:
            categories = json.loads(categories)
        except:
            categories = [cat.strip() for cat in categories.split(',')] if ',' in categories else [categories]

    if not isinstance(categories, list):
        categories = [str(categories)]

    return categories


In [ ]:
def extract_address_features(address):
    address = str(address).lower()
    features = {}

    street_types = ['st', 'ave', 'blvd', 'dr', 'ln', 'rd', 'way', 'pkwy', 'pl']
    for st_type in street_types:
        features[f'addr_type_{st_type}'] = 1 if f" {st_type}" in address or f" {st_type}." in address else 0

    directions = ['north', 'south', 'east', 'west', 'n', 's', 'e', 'w', 'ne', 'nw', 'se', 'sw']
    for direction in directions:
        features[f'addr_dir_{direction}'] = 1 if direction in address.split() else 0

    numbers = re.findall(r'\d+', address)
    features['addr_number'] = int(numbers[0]) if numbers else -1

    return features


In [ ]:
train_df = train_reducido
test_df = test_completo.copy()

In [ ]:
train_df = generar_embeddings(train_df, model)


Generando embeddings para texto...


In [ ]:
test_df = generar_embeddings(test_df, model)

Generando embeddings para texto...


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,LeakyReLU, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
import ast
import os

# Extraer los embeddings como características
X_train_embeddings = np.array(train_df['text_embedding'].tolist())
X_test_embeddings = np.array(test_df['text_embedding'].tolist())

# Seleccionar características numéricas adicionales
numerical_features = [
    'useful', 'funny', 'cool',              # Características de la reseña
    'review_count', 'useful_usuario', 'funny_usuario', 'cool_usuario',
    'fans', 'average_stars', 'años_en_yelp', 'num_amigos',
    'total_compliments', 'ratio_useful', 'ratio_funny', 'ratio_cool', 'popularidad',  # Usuario
    'num_reviews_negocio', 'is_open', 'num_categorias',  # Negocio
    'latitude', 'longitude'                 # Ubicación
]

# Verificar que las características numéricas existan en ambos conjuntos
valid_features = [feat for feat in numerical_features if feat in train_df.columns and feat in test_df.columns]
print(f"Características numéricas utilizadas: {valid_features}")

# Preparar características numéricas
X_train_numerical = train_df[valid_features].fillna(0).copy()
X_test_numerical = test_df[valid_features].fillna(0).copy()

# Normalizar características numéricas
scaler = StandardScaler()
X_train_numerical_scaled = scaler.fit_transform(X_train_numerical)
X_test_numerical_scaled = scaler.transform(X_test_numerical)

# Combinar embeddings con características numéricas
X_train = np.hstack((X_train_embeddings, X_train_numerical_scaled))
X_test = np.hstack((X_test_embeddings, X_test_numerical_scaled))

# Variable objetivo
y_train = train_df['estrellas_review'].values

# Dividir conjunto de entrenamiento para validación
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)

# Definir la arquitectura de la red neuronal
print("Construyendo el modelo...")
model = Sequential([
    Input(shape=(X_train.shape[1],)),  # input shape como capa inicial

    Dense(512),
    LeakyReLU(alpha=0.01),
    Dropout(0.4),

    Dense(256),
    LeakyReLU(alpha=0.01),
    Dropout(0.3),

    Dense(128),
    LeakyReLU(alpha=0.01),
    Dropout(0.2),

    Dense(64),
    LeakyReLU(alpha=0.01),
    Dropout(0.15),

    Dense(32),
    LeakyReLU(alpha=0.01),
    Dense(1)  # salida regresión
])

# Compilar el modelo
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mae'])

# Definir early stopping para evitar sobreajuste
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Entrenar el modelo
print("Entrenando el modelo...")
history = model.fit(
    X_train_split, y_train_split,
    epochs=30,
    batch_size=256,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)

# Evaluar el modelo en el conjunto de validación
val_loss, val_mae = model.evaluate(X_val, y_val)
print(f"Pérdida en validación: {val_loss:.4f}")
print(f"MAE en validación: {val_mae:.4f}")

# Hacer predicciones en el conjunto de prueba
print("Realizando predicciones...")
predictions = model.predict(X_test)

# Asegurarse de que las predicciones estén en el rango correcto (entre 1 y 5)
predictions = np.clip(predictions, 1, 5)

# Crear un DataFrame con los resultados
results_df = pd.DataFrame({
    'review_id': test_df['review_id'],
    'stars': predictions.flatten()
})

# Guardar resultados en CSV
output_file = 'predictions_DL.csv'
results_df.to_csv(output_file, index=False)
print(f"Predicciones guardadas en {output_file}")

# Mostrar las primeras predicciones
print("\nPrimeras 5 predicciones:")
print(results_df.head())

Características numéricas utilizadas: ['useful', 'funny', 'cool', 'review_count', 'useful_usuario', 'funny_usuario', 'cool_usuario', 'fans', 'average_stars', 'años_en_yelp', 'num_amigos', 'total_compliments', 'ratio_useful', 'ratio_funny', 'ratio_cool', 'popularidad', 'num_reviews_negocio', 'is_open', 'num_categorias', 'latitude', 'longitude']
Construyendo el modelo...


/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Entrenando el modelo...
Epoch 1/30
681/681 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - loss: 2.1489 - mae: 1.0646 - val_loss: 0.8829 - val_mae: 0.7763
Epoch 2/30
681/681 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.7226 - mae: 0.6393 - val_loss: 0.8308 - val_mae: 0.7501
Epoch 3/30
681/681 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.6340 - mae: 0.5843 - val_loss: 0.6685 - val_mae: 0.6538
Epoch 4/30
681/681 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5853 - mae: 0.5528 - val_loss: 0.6147 - val_mae: 0.6186
Epoch 5/30
681/681 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.5580 - mae: 0.5352 - val_loss: 0.5869 - val_mae: 0.5916
Epoch 6/30
681/681 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.5361 - mae: 0.5213 - val_loss: 0.5766 - val_mae: 0.5891
Epoch 7/30
681/681 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.5173 - mae: 0.5111 - val_loss: 0.5451 - val_mae: 0.5175
Epoch 8/30
681/681 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.5052 - mae: 0.5033 - val_loss: 0.5542 - val_mae: 0.5547
Epoch 9/30
681/681 ━━━━━━━━━━━━